# Base R — Technical Reference

R companion to the NumPy reference. Where NumPy has `ndarray`, base R has
**vectors, matrices, and arrays** — same core ideas (shape, dtype, vectorized
ops, broadcasting), different defaults (1-indexed, copy-on-modify instead of
views, recycling instead of broadcasting).

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Vector / matrix creation | Build vectors, matrices, arrays from scratch or data | §1 |
| Properties & types | Inspect shape, type, memory | §2 |
| Indexing & slicing | Subset vectors / matrices / data frames | §3 |
| Reshaping & combining | Change dimensions, combine objects | §4 |
| Math & aggregations | Element-wise ops, reductions | §5 |
| Recycling (R's "broadcasting") | Operations on objects of different lengths | §6 |
| Linear algebra | Matrix ops, decompositions, solvers | §7 |
| Random & simulation | Sampling, seeding, distributions | §8 |
| Performance | Copy-on-modify, vectorization | §9 |

---
## When to Use

| Signal | R function to reach for |
| :--- | :--- |
| Create a vector of zeros / ones | `numeric(n)`, `rep(1, n)`, `rep(7, n)` |
| Evenly spaced values | `seq(from, to, by=)` (step-based) or `seq(from, to, length.out=)` (count-based) |
| Filter elements by condition | Logical indexing `x[x > 0]` |
| Apply condition element-wise | `ifelse(cond, x, y)` |
| Aggregate across a margin | `rowSums()`, `colSums()`, `apply(m, 1, fn)` / `apply(m, 2, fn)` |
| Reshape without copying conceptually | `dim(a) <- c(r, c)` or `t()` |
| Stack arrays side by side | `cbind()` |
| Stack arrays top to bottom | `rbind()` |
| Matrix multiplication | `A %*% B` |
| Solve linear system Ax = b | `solve(A, b)` |
| Generate reproducible random data | `set.seed()` then `r*()` family |
| Check if a modification copies | R always copies on modify — there is no in-place view to check |

---
## §1 — Vector & Matrix Creation

Every R workflow starts here. Know which constructor to reach for and what type it produces by default.

In [ ]:
# From R literals
c(1, 2, 3)                          # numeric vector, type inferred (double)
matrix(c(1, 2, 3, 4), nrow = 2)     # 2x2 matrix, filled COLUMN-major by default
c(1L, 2L, 3L)                       # integer vector (force with L suffix)

# Filled vectors / matrices
numeric(12)                         # vector of 12 zeros (double)
rep(1, 12)                          # vector of 12 ones
rep(7, 12)                          # vector filled with 7
matrix(0, nrow = 3, ncol = 4)       # 3x4 matrix of 0.0
matrix(1, nrow = 3, ncol = 4)       # 3x4 matrix of 1.0
matrix(7, nrow = 3, ncol = 4)       # 3x4 matrix filled with 7
diag(4)                             # 4x4 identity matrix
matrix(nrow = 3, ncol = 4)          # uninitialized -> filled with NA (R has no "garbage" array)

# Range vectors
seq(0, 10, by = 2)                  # 0 2 4 6 8 10 -- like Python range(), step-based, end INCLUSIVE if reached
seq(0, 1, length.out = 5)           # 0 0.25 0.5 0.75 1.0 -- N evenly spaced points

# From an existing vector/matrix
arr <- matrix(1:12, nrow = 3)
matrix(0, nrow = nrow(arr), ncol = ncol(arr))   # same shape as arr, filled with 0
arr * 0                                          # same shape and type, filled with 0 -- idiomatic shortcut
identical(arr, arr)                              # `arr` is already a value; R copies on assignment, see SS9

**`seq(..., by=)` vs `seq(..., length.out=)`:**

| | Control | Endpoint included | Use when |
| :--- | :--- | :--- | :--- |
| `seq(from, to, by = step)` | Step size | Only if reached exactly | You know the step |
| `seq(from, to, length.out = n)` | Count | Always | You know how many points |

**Common mistakes:**
- `matrix(1:6, 3, 4)` silently recycles/truncates if the element count doesn't divide evenly into the shape — R warns but still runs
- `seq()` with float steps can accumulate floating point error, same as `np.arange` — use `length.out` for evenly spaced floats
- `matrix(nrow=, ncol=)` with no data fills with `NA`, not arbitrary memory garbage like `np.empty` -- this is actually safer than NumPy's equivalent
- R's default `matrix()` fill order is **column-major**; pass `byrow = TRUE` to fill row-major like NumPy's default

---
## §2 — Properties & Types

Always check shape and type before operating. Silent type coercion and shape mismatches are the most common source of bugs, exactly as in NumPy.

In [ ]:
a <- matrix(1:6, nrow = 2, ncol = 3, byrow = TRUE)

# Shape and dimensions
dim(a)            # c(2, 3) -- (rows, cols)
length(dim(a))    # 2 -- number of dimensions
length(a)         # 6 -- total number of elements (NumPy's .size)
nrow(a)           # 2 -- length of first dimension only (like Python's len())

# Type
typeof(a)         # "integer"
class(a)          # "matrix" "array"
storage.mode(a) <- "double"   # cast to double (R's float64) -- modifies via assignment, not a method call
as.integer(a)     # cast to integer (flattens to a vector -- see SS4 for reshape-safe casting)

# Memory
object.size(a)    # total bytes (approx, includes R object overhead)

# Common types
# double  -- default for floats, highest precision (NumPy's float64)
# integer -- whole numbers, smaller footprint (suffix literals with L)
# logical -- TRUE/FALSE (NumPy's bool)
# character -- strings
# complex -- complex numbers
# R has NO direct float32 / int32 equivalent in base types -- see the `bit64`/`float` packages if needed

**Common mistakes:**
- Mixing integer and double vectors -- R upcasts silently: `c(1L, 2L) + c(0.5)` returns `double`, same idea as NumPy's int->float upcast
- Using `nrow(a)` thinking it is total count -- it is rows only; use `length(a)` for the NumPy `.size` equivalent
- Forgetting `storage.mode(a) <- "double"` modifies in place via the replacement function -- unlike `a.astype()` in NumPy, R's "casting" functions like `as.integer()` do NOT preserve dimensions automatically (see SS4)

---
## §3 — Indexing & Slicing

R indexing is **1-based**, and unlike NumPy, indexing a vector or matrix in R **always copies** (R has copy-on-modify semantics, not views). See SS9 for what this means for performance.

In [ ]:
a <- matrix(1:9, nrow = 3, byrow = TRUE)

# 1D indexing (treating the matrix as a vector, or indexing a plain vector)
a[1, ]            # first row
a[nrow(a), ]      # last row (no negative-index-from-end shortcut like Python's -1)
a[2:3, ]          # rows 2 and 3 (end INCLUSIVE, unlike Python's end-exclusive)
a[c(TRUE, FALSE), ]  # every other row -- logical vector recycles across rows

# 2D indexing -- [row, col]
a[2, 3]           # row 2, col 3 -> 6
a[1, ]            # entire first row -> c(1, 2, 3)
a[, 2]            # entire second column -> c(2, 5, 8)
a[1:2, 2:3]       # submatrix: rows 1-2, cols 2-3

# Logical indexing -- filter elements by condition
a[a > 5]                         # -> c(6, 7, 8, 9) (1D result, column-major flattened)
a[(a > 2) & (a < 8)]             # AND condition -- same `&` operator, but R requires no extra parens caveat (still recommended)
rows <- a[, 1] > 3                # logical mask on first column
a[rows, ]                         # select rows where first col > 3

# Index-vector ("fancy") indexing -- select with a vector of positions
a[c(1, 3), ]                      # rows 1 and 3
a[, c(1, 3)]                      # columns 1 and 3
a[cbind(c(1, 2), c(2, 3))]        # elements (1,2) and (2,3) -> c(2, 6) -- matrix-of-indices form

# ifelse -- conditional element selection (vectorized, same shape as input)
which(a > 5, arr.ind = TRUE)      # returns a matrix of (row, col) positions, NumPy's np.where(cond) equivalent
ifelse(a > 5, a, 0)               # if > 5 keep value, else 0 (same shape as a)
ifelse(a > 5, "high", "low")      # string labels

**Views vs copies -- the single biggest conceptual difference from NumPy:**

| Operation | NumPy | R |
| :--- | :--- | :--- |
| Basic slicing | View (shares memory) | **Copy** (always) |
| Logical indexing | Copy | Copy |
| Index-vector indexing | Copy | Copy |

R has **no view concept** for ordinary indexing -- every `a[...]` read produces an independent copy, and modifying that copy never touches `a`. This removes an entire category of NumPy bugs (accidentally mutating an array via a slice) but means you cannot rely on slicing being free; see SS9 for the actual cost model (copy-on-*modify*, not copy-on-*read*).

**Common mistakes:**
- Assuming 0-based indexing out of NumPy habit -- `a[0, ]` in R either errors or returns an empty result depending on context; R starts at 1
- Forgetting that `a[2:3, ]` includes both endpoints -- R ranges are inclusive on both ends, unlike Python slices
- `which(cond)` alone returns flat positions, not a value-substitution result -- use `ifelse(cond, x, y)` for value substitution, same caveat as `np.where(cond)` vs `np.where(cond, x, y)`

---
## §4 — Reshaping & Combining

Reshaping in R is done via the `dim<-` replacement function (in place, no copy concept to reason about since R always copies on write). Use `cbind`/`rbind` to combine objects along existing or new axes.

In [ ]:
a <- 0:11   # 0, 1, ..., 11 (12 elements)

# Reshape
matrix(a, nrow = 3, ncol = 4)              # 3 rows x 4 cols, column-major fill -- total elements must match
matrix(a, nrow = 3)                        # ncol inferred automatically (R's equivalent of -1)
matrix(a, ncol = 1)                        # column vector: (12, 1)
matrix(a, nrow = 1)                        # row vector: (1, 12)

# Transpose
m <- matrix(a, nrow = 3, ncol = 4)
t(m)                                       # (4, 3) -- swaps rows/cols, only 2D (no axis-order arg like np.transpose for nD)
aperm(array(a, dim = c(3, 4)), c(2, 1))    # nD generalization of transpose -- explicit axis order

# Flatten -- always a copy in R (consistent with SS3 -- nothing is ever a view)
as.vector(m)                               # (12,) 1D copy, column-major order
c(m)                                       # equivalent shorthand

# Add / remove dimensions
array(a, dim = c(1, 12))                   # (1, 12) -- add axis at position 0
array(a, dim = c(12, 1))                   # (12, 1) -- add axis at position 1
drop(matrix(a, nrow = 1))                  # remove size-1 dimensions -> (12,)

# Combining
x <- c(1, 2, 3)
y <- c(4, 5, 6)

rbind(x, y)                                # stack as rows -> shape (2, 3)
c(x, y)                                    # concatenate 1D -> shape (6,) -- R's hstack-for-vectors equivalent
cbind(x, y)                                # stack 1D vectors as columns -> shape (3, 2)
abind::abind(x, y, along = 1)              # general n-D concatenate along any axis (needs the `abind` package)
array(c(x, y), dim = c(2, 3))              # new-axis stack -> shape (2, 3), R's np.stack equivalent

# Split
split(a, rep(1:3, length.out = length(a)))   # split into 3 groups (round-robin, not contiguous like np.split)
matrix(a, nrow = 3) |> split(row(matrix(a, nrow = 3)))  # contiguous row-wise split, closer to np.split semantics

**`rbind`/`cbind` vs `c()` vs `abind::abind`:**

| | Creates new axis? | Use when |
| :--- | :--- | :--- |
| `rbind` / `cbind` | Yes (for vectors), No (for matrices of compatible shape) | Combining 1D vectors into a 2D matrix, or stacking matrices |
| `c()` | No | Combining along the existing (only) axis of plain vectors |
| `abind::abind(..., along=)` | Depends on `along` | nD generalization, R's `np.stack` / `np.concatenate` equivalent |

**Common mistakes:**
- `matrix(a, nrow=, ncol=)` recycles silently with a warning if the element count doesn't match -- always check `length(a)` before reshaping, same discipline as checking `a.size` in NumPy
- R fills matrices **column-major** by default; NumPy fills **row-major** by default -- always double check with `byrow = TRUE` when porting layouts from Python
- `as.vector()`/`c()` always copy, matching `np.flatten()`'s always-copy behavior -- there is no `ravel()`-style "view if possible" option in R, because nothing is ever a view

**`rep()` variants for the NumPy `repeat`/`tile` distinction:**

| | What it does | NumPy equivalent |
| :--- | :--- | :--- |
| `rep(x, each = n)` | repeat **each element** n times | `np.repeat(x, n)` |
| `rep(x, times = n)` | repeat the **whole vector** n times | `np.tile(x, n)` |

In [ ]:
x <- c(1, 2, 3)
cat("rep(x, each=2) :", rep(x, each = 2), "\n")   # each element
cat("rep(x, times=2):", rep(x, times = 2), "\n")  # whole vector

---
## §5 — Math & Aggregations

All operations are element-wise by default, exactly as in NumPy. Use `apply()`, `rowSums()`/`colSums()`, or `rowMeans()`/`colMeans()` to reduce along a specific margin.

In [ ]:
a <- matrix(1:6, nrow = 2, byrow = TRUE)

# Element-wise arithmetic
a + 10                           # add scalar to every element
a * 2                            # multiply every element
a ^ 2                            # square every element
sqrt(a)                          # square root element-wise
exp(a)                           # e^x element-wise
log(a)                           # natural log element-wise
abs(a)                           # absolute value

# Aggregations -- default: reduce entire object to scalar
sum(a)                           # sum of all elements
mean(a)                          # mean of all elements
sd(a)                            # standard deviation (NOTE: R divides by n-1, NumPy's .std() divides by n -- see callout below)
min(a)
max(a)
which.min(a)                     # index of minimum element (flattened, column-major)
which.max(a)                     # index of maximum element (flattened, column-major)

# Aggregations along a margin
colSums(a)                       # sum down rows -> result length 3 -- one value per column (NumPy axis=0)
rowSums(a)                       # sum across cols -> result length 2 -- one value per row (NumPy axis=1)
rowMeans(a)                      # one mean per row, always returns a plain vector (R has no keepdims concept)

# Cumulative
t(apply(a, 1, cumsum))           # running total along each row (apply transposes results -- the `t()` undoes that)
apply(a, 2, cumprod)             # running product down each column

# Matrix operations
A <- matrix(c(1, 2, 3, 4), nrow = 2, byrow = TRUE)
B <- matrix(c(5, 6, 7, 8), nrow = 2, byrow = TRUE)

A * B                            # element-wise multiplication (NOT matrix multiply)
A %*% B                          # matrix multiplication -- R's equivalent of NumPy's @
crossprod(A, B)                  # t(A) %*% B, computed more efficiently
outer(as.vector(A), as.vector(B))  # outer product

**Margin mental model (R's `MARGIN` argument, same idea as NumPy's `axis`):**
- `MARGIN = 1` (rows) -> apply/reduce **across columns for each row** -- conceptually NumPy's `axis=1`
- `MARGIN = 2` (columns) -> apply/reduce **down rows for each column** -- conceptually NumPy's `axis=0`
- This is the **opposite numbering convention** from NumPy's axis -- the single most common source of bugs when porting code

**Common mistakes:**
- `A * B` is element-wise, not matrix multiplication -- use `A %*% B` for matrix multiply, same trap as NumPy's `A * B` vs `A @ B`
- R's `sd()`/`var()` use the **sample** formula (divide by n-1); NumPy's `.std()`/`.var()` default to the **population** formula (divide by n) -- pass `ddof=1` in NumPy or compute manually in R to match
- R's `MARGIN=1`/`MARGIN=2` in `apply()` is the reverse of NumPy's `axis=0`/`axis=1` -- always sanity-check against a known small example when porting

**`%*%` vs `crossprod`/`tcrossprod` vs `*`:**

| | 1-D inputs | 2-D inputs | Use when |
| :--- | :--- | :--- | :--- |
| `A %*% B` | dot product (1x1 matrix) | matrix multiply | General matrix multiplication |
| `crossprod(A, B)` | -- | `t(A) %*% B`, faster | You need `A'B` -- common in regression (`X'X`) |
| `tcrossprod(A, B)` | -- | `A %*% t(B)`, faster | You need `AB'` |

**`identical()` vs `all.equal()` vs `==`:**

| | Returns | Float-safe? | Use when |
| :--- | :--- | :--- | :--- |
| `a == b` | elementwise logical vector | No | You want a per-element mask |
| `identical(a, b)` | single logical, **exact** | No | Exact equality including type/attributes |
| `isTRUE(all.equal(a, b))` | single logical, **tolerance** | Yes | Comparing floats (always use this) |

In [ ]:
# float comparison trap -- same IEEE-754 issue as NumPy
a <- c(0.1 + 0.2, 1.0)
b <- c(0.3, 1.0)
cat("a == b              :", a == b, "\n")
cat("identical(a, b)     :", identical(a, b), "\n")
cat("isTRUE(all.equal)   :", isTRUE(all.equal(a, b)), "\n")

---
## §6 — Recycling (R's "Broadcasting")

R does not call this "broadcasting", but the mechanism -- stretching a shorter object to match a longer one without copying data -- serves the same purpose. The rules are looser and less explicit than NumPy's, which is exactly why this section exists: recycling will *silently* do something unintended where NumPy would raise a `ValueError`.

In [ ]:
# Recycling rules:
# 1. The shorter vector is repeated ("recycled") to match the length of the longer one
# 2. If the longer length is NOT a multiple of the shorter, R recycles anyway and issues a WARNING (not an error)
# 3. There is no NumPy-style "shape compatibility" check for plain vector + vector ops

# Scalar broadcast -- always works, same as NumPy
a <- matrix(1:6, nrow = 2, byrow = TRUE)    # shape (2, 3)
a + 10                                       # 10 recycles to every element

# Vector recycled against matrix -- R recycles COLUMN-major, this is the trap
row <- c(10, 20, 30)                         # length 3
a + row                                      # recycles down columns first, NOT "add row to each row" like NumPy!
# To genuinely add a row-vector to each row (NumPy's intuitive behavior), transpose twice:
t(t(a) + row)                                # correct: adds `row` to every row of `a`

col <- c(10, 20)                             # length 2, matches nrow(a)
a + col                                      # this one recycles correctly down each column (R's natural fill order)

# Common pattern: normalize each row (subtract row mean) -- the safe, explicit way
sweep(a, MARGIN = 1, STATS = rowMeans(a), FUN = "-")   # purpose-built for this -- avoids the recycling trap entirely

# Outer product via outer() -- R's explicit, safe alternative to broadcasting tricks
x <- c(1, 2, 3)                              # length 3
y <- c(10, 20)                                # length 2
outer(y, x)                                   # (2,3) result -- explicit, no recycling ambiguity

# Shape "compatibility" in R -- there is no error, only a warning, for mismatched lengths:
# length 6 vector + length 3 -> OK, recycled exactly twice, no warning
# length 6 vector + length 4 -> recycled with a WARNING ("longer object length is not a multiple...")
# this is the single biggest silent-bug risk ported from NumPy's stricter shape rules

**Recycling vs NumPy broadcasting -- the critical difference:**

| | NumPy | R |
| :--- | :--- | :--- |
| Mismatched, non-multiple lengths | Raises `ValueError` | Recycles anyway, only a **warning** |
| Matrix + short vector direction | Explicit per-axis rules (`axis=` aware) | Always recycles **column-major**, regardless of intent |
| Safe way to add a row vector to every row | `a + row` just works | Must use `sweep()` or `t(t(a) + row)` |
| Safe way to add a column vector to every column | `a + col[:, None]` | `a + col` (matches R's natural column-major recycling) |

**Common mistakes:**
- Assuming `matrix + row_vector` adds the vector to each row like NumPy -- R recycles column-major, so it actually adds the vector down each column first, only "looking right" by coincidence for square-ish cases. **Always use `sweep()` when intent matters.**
- Trusting that R will error on a shape mismatch -- it almost never does; mismatched-but-not-multiple lengths silently recycle with only a warning that is easy to miss in long output
- Forgetting that `outer(x, y)` (note argument order) builds an outer product explicitly -- prefer it over manual `newaxis`-style tricks for clarity

---
## §7 — Linear Algebra

All linear algebra is available in base R (no separate `linalg` submodule needed). Critical for quantitative analyst roles and statistics. Always check that your matrix is square / full-rank before inverting.

In [ ]:
A <- matrix(c(2, 1, 5, 3), nrow = 2, byrow = TRUE)   # a GENERAL (non-symmetric) matrix
b <- c(4, 7)
S <- A %*% t(A)                       # build a SYMMETRIC, positive-definite matrix for eigen/chol

# Basic matrix properties
det(A)                                # determinant -- 0 means singular (not invertible)
qr(A)$rank                            # rank (via QR decomposition -- base R has no single matrix_rank() function)
sum(diag(A))                          # trace -- sum of diagonal elements (no built-in trace())
diag(A)                               # extract diagonal as 1D vector
diag(c(1, 2, 3))                      # create diagonal matrix from 1D vector

# Inverse
solve(A)                              # A^-1 -- only for square, non-singular matrices

# Solve linear system Ax = b -- same function name, dual purpose (preferred over solve(A) %*% b, more numerically stable)
x <- solve(A, b)                      # finds x such that Ax = b

# Norms
sqrt(sum(b^2))                        # L2 norm (Euclidean) of a vector -- base R has no norm() for plain vectors
norm(A, type = "F")                   # Frobenius norm of a matrix
sum(abs(b))                           # L1 norm

# Eigenvalues and eigenvectors -- ONE function handles both general and symmetric in R
eig_general <- eigen(A)               # GENERAL matrix -- works for any square matrix
eig_sym <- eigen(S, symmetric = TRUE) # tell R it's symmetric for a faster, more stable algorithm

# Singular Value Decomposition -- A = U %*% diag(d) %*% t(V)
sv <- svd(A)                          # sv$d is the 1D vector of singular values (descending); sv$u, sv$v are the factors

# Least squares -- solve over-determined system (more equations than unknowns)
fit <- lm.fit(A, b)                   # base R's least-squares solver (or use lm() for full regression objects)

# Cholesky decomposition -- t(L) %*% L = S in R's convention (upper-triangular by default, opposite of NumPy's lower-triangular)
L <- chol(S)                          # requires SYMMETRIC positive-definite -- pass S, not A

**`solve(A, b)` vs `solve(A) %*% b`:**

| | Speed | Numerical stability | Use when |
| :--- | :--- | :--- | :--- |
| `solve(A, b)` | Faster | Better | Solving Ax = b |
| `solve(A) %*% b` | Slower | Worse | Only when you need A^-1 explicitly |

**`eigen()` general vs `symmetric = TRUE`:**

| | Valid input | Output | Use when |
| :--- | :--- | :--- | :--- |
| `eigen(A)` (default) | Any square matrix | Possibly complex, R auto-detects symmetry by checking the matrix | General matrix |
| `eigen(S, symmetric = TRUE)` | You assert symmetric/Hermitian | Real, faster algorithm | You already know it's symmetric -- skip R's auto-check for speed |

Unlike NumPy (separate `eig` vs `eigh` functions, where `eigh` silently returns a wrong answer on non-symmetric input), R's `eigen()` **auto-detects** symmetry by default and dispatches accordingly -- the `symmetric=` argument is purely a speed optimization, not a correctness requirement. This is a meaningfully safer default than NumPy's.

**Common mistakes:**
- Calling `solve(A)` on a singular matrix -- raises an error (`Lapack routine dgesv: system is exactly singular`); check `det(A) != 0` first, same discipline as NumPy
- R's `chol()` returns an **upper**-triangular factor (`t(L) %*% L = S`) by default, the transpose convention of NumPy's `np.linalg.cholesky` (which returns lower-triangular, `L @ L.T = S`) -- transpose if you need the NumPy convention
- Forgetting that `svd()` returns `$v` (not transposed), while NumPy's `svd` returns `Vt` (transposed) -- `A ≈ sv$u %*% diag(sv$d) %*% t(sv$v)` in R vs `A ≈ U @ diag(S) @ Vt` in NumPy

**Verified -- why `eigen()`'s auto-detection matters:**

In [ ]:
A <- matrix(c(2, 1, 5, 3), nrow = 2, byrow = TRUE)   # non-symmetric
S <- A %*% t(A)                                       # symmetric, positive-definite

cat("S symmetric?       :", isSymmetric(S), "\n")
cat("eigen(S)$values     :", sort(eigen(S)$values), "\n")              # correct
cat("eigen(S, sym=T)$val :", eigen(S, symmetric = TRUE)$values, "\n")  # correct (R's eigen sorts descending by default)
cat("eigen(A)$values      :", sort(eigen(A)$values), "\n")             # correct for general A -- R auto-detects A is NOT symmetric
# Unlike NumPy's eigh(), there is no "wrong answer" failure mode here -- eigen() checks symmetry itself unless told otherwise

---
## §8 — Random & Simulation

Use `set.seed()` for reproducibility. Unlike NumPy's modern `default_rng()` generator-object API, R's random functions are all driven by **global RNG state** -- there is no first-class "local generator" equivalent in base R (the `rngtools`/`dqrng` packages add one if you need it).

In [ ]:
set.seed(42)                              # set the global seed -- R's only API, no generator object

matrix(runif(12), nrow = 3, ncol = 4)     # uniform [0, 1) -- shape (3, 4)
matrix(sample(0:9, 12, replace = TRUE), nrow = 3, ncol = 4)  # integers in [0, 10)
rnorm(100, mean = 0, sd = 1)              # normal distribution mu=0, sigma=1
runif(100, min = 0, max = 1)              # uniform distribution
rbinom(100, size = 10, prob = 0.5)        # binomial distribution
sample(1:4, size = 10, replace = TRUE)    # sample with replacement
sample(1:4, size = 3, replace = FALSE)    # sample without replacement
arr <- 1:10
sample(arr)                                # shuffled COPY -- R has no in-place shuffle (everything copies, see SS9)
sample(arr, length(arr))                   # equivalent, explicit form

# Monte Carlo example -- estimate pi
set.seed(0)
n <- 1e6
x <- runif(n); y <- runif(n)
inside <- (x^2 + y^2) < 1
pi_estimate <- 4 * mean(inside)            # approx 3.1416

**R's global-state RNG vs NumPy's `default_rng()` generator objects:**

| | `np.random.seed()` (legacy) | `np.random.default_rng()` (modern) | R's `set.seed()` |
| :--- | :--- | :--- | :--- |
| Thread-safe | No | Yes | No (same limitation as NumPy's legacy API) |
| Reproducible | Yes (global state) | Yes (local state) | Yes (global state) |
| Local/isolated generator object | No | Yes | Not in base R |

**Common mistakes:**
- Expecting an R equivalent of `rng.shuffle()` that mutates in place -- R has none; `sample(x)` always returns a new shuffled vector, consistent with R's copy-on-modify model throughout
- `sample(0:9, ...)` -- R's `sample()` range is inclusive on both ends by default (`0:9` is 0 through 9), so match the NumPy convention deliberately when porting `rng.integers(0, 10)` (which excludes 10)
- Running parallel R processes (e.g. via `parallel::mclapply`) without `set.seed()`'s `RNGkind("L'Ecuyer-CMRG")` stream-safe mode -- same race-condition class of bug as NumPy's legacy global-state API in threaded code

---
## §9 — Performance

R's speed, like NumPy's, comes from vectorized operations implemented in C under the hood. The moment you introduce an explicit `for` loop over elements, you lose that advantage -- but R's memory model differs from NumPy's in one fundamental way: **everything copies on modify**, there are no views.

In [ ]:
# Copy-on-modify -- the core R memory concept (replaces NumPy's views vs copies distinction)
a <- matrix(1:12, nrow = 3, ncol = 4)
b <- a[1:2, ]                     # looks like NumPy's "view", but is NOT -- modifying b never touches a
c_ <- a[1:2, ]                     # genuinely independent too -- in R there is no operation that returns a view
b[1, 1] <- 99
identical(a[1, 1], b[1, 1])        # FALSE -- proof that b was never linked to a in the first place
# R's actual optimization is "copy-on-write": the copy is deferred until the first modification,
# so reading a[1:2, ] is cheap even though it is conceptually always a copy

# Vectorization -- always prefer over loops, exactly as in NumPy
a <- 1:1e6

# Slow -- explicit R loop
result <- numeric(length(a))
for (i in seq_along(a)) result[i] <- a[i]^2   # ~200ms-equivalent penalty

# Fast -- vectorized
result <- a ^ 2                                # ~1ms-equivalent -- 100-200x faster

# Prefer vectorized built-ins over sapply/loops for math
sqrt(a)                            # fast -- C implementation under the hood
log(a)                             # fast
sapply(a, sqrt)                    # slow -- still loops internally despite the tidy syntax

# Memory layout -- R is COLUMN-major (opposite of NumPy's default row-major C order)
# iterating/aggregating over columns is the cache-friendly direction in R
a <- matrix(1, nrow = 1000, ncol = 1000)
colSums(a)                         # faster -- iterates down columns (R's natural storage order)
rowSums(a)                         # slightly slower -- iterates across rows (against the grain)

# R has no native float32 -- doubles (float64) are the only floating type in base R
a <- matrix(1, nrow = 1000, ncol = 1000)
object.size(a)                     # always sized as if float64 -- the `float` package adds true float32 support if needed

# Compact notation for contractions -- R has no single einsum equivalent;
# express common contractions directly, or reach for the `einsum` package for the general case
A %*% B                            # matrix multiply (the einsum 'ij,jk->ik' case)
rowSums(X * Y)                     # batch dot product per row (the einsum 'bi,bi->b' case)

**Common mistakes:**
- Calling `.copy()`-equivalent operations defensively, the way you might in NumPy -- unnecessary in R, since every assignment is already copy-on-modify by default; there is nothing to "protect" against
- Using `sapply()`/`for` loops inside hot paths instead of vectorized operations -- same speed penalty class as Python `math` calls inside a loop instead of NumPy ufuncs
- Building large matrices and assuming float32-equivalent memory savings are available -- base R doubles are always 8 bytes; reach for the `float` package if true float32 storage matters
- Growing a vector inside a loop with `c(result, new_value)` -- this **reallocates and copies the entire vector every iteration**, an R-specific trap with no direct NumPy analog (NumPy users coming from `np.append()` in a loop have the closest intuition for why this is slow); pre-allocate with `numeric(n)` instead, as shown above

---
# Decision Guide

```
Creating a vector or matrix?
+-- From existing data                        -> c(), matrix()
+-- All zeros / ones / constant                -> numeric(n), rep(1, n), rep(k, n)
+-- Range with fixed step                      -> seq(from, to, by = step)
+-- N evenly spaced points                     -> seq(from, to, length.out = n)
+-- Identity matrix                            -> diag(n)

Indexing / filtering?
+-- Slice by position                          -> a[start:stop, ] -- ALWAYS a copy (no views in R)
+-- Filter by condition                        -> a[a > threshold] -- copy
+-- Select by index vector                     -> a[c(1, 3, 5), ] -- copy
+-- Conditional value substitution              -> ifelse(cond, x, y)
+-- Need to modify without affecting original  -> nothing extra needed -- it already won't

Changing shape?
+-- Reshape to known dimensions                -> matrix(a, nrow = r, ncol = c)
+-- Transpose                                  -> t(a)
+-- Flatten to 1D                              -> as.vector(a) / c(a) -- always a copy
+-- Add a dimension                            -> array(a, dim = c(1, n)) or array(a, dim = c(n, 1))

Combining objects?
+-- Stack as new rows                          -> rbind(a, b)
+-- Stack as new columns                       -> cbind(a, b)
+-- Along a new axis (nD)                      -> abind::abind(a, b, along = n)
+-- Along an existing axis (1D)                -> c(a, b)

Aggregating?
+-- Entire object                              -> sum(a), mean(a), etc.
+-- One value per column                       -> colSums(a) / colMeans(a)  (NumPy axis=0)
+-- One value per row                          -> rowSums(a) / rowMeans(a)  (NumPy axis=1)
+-- Custom reduction along a margin            -> apply(a, MARGIN, fn)  -- MARGIN is REVERSED vs NumPy's axis!

Linear algebra?
+-- Matrix multiplication                      -> A %*% B
+-- Solve Ax = b                               -> solve(A, b)
+-- Eigenvalues (any matrix)                   -> eigen(A)  -- auto-detects symmetry
+-- Eigenvalues (symmetric, faster)            -> eigen(A, symmetric = TRUE)
+-- SVD                                        -> svd(A)
+-- Inverse (use sparingly)                    -> solve(A)

Performance?
+-- Slow explicit loop over a vector           -> replace with a vectorized op
+-- Growing a vector inside a loop             -> pre-allocate with numeric(n) first
+-- Need genuinely independent data            -> nothing extra -- R already copies on modify
+-- Complex multi-axis contraction              -> rowSums(X * Y) patterns, or the `einsum` package
```